# ਪਾਠ 12 - ਏਜੰਟ ਸਕ੍ਰੈਚਪੈਡ ਨਾਲ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਘਟਾਉਣਾ

ਇਹ ਨੋਟਬੁੱਕ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਲੰਮੀ ਗੱਲਬਾਤਾਂ ਵਿੱਚ ਸੰਦਰਭ ਪ੍ਰਬੰਧਨ ਨੂੰ ਕਿਵੇਂ ਸੰਭਾਲਣਾ ਹੈ, ਇਹ ਦਰਸਾਉਂਦਾ ਹੈ। ਜਿਵੇਂ ਜਿਵੇਂ ਗੱਲਬਾਤ ਵੱਡੀ ਹੁੰਦੀ ਹੈ, ਟੋਕਨ ਗਿਣਤੀ ਵੱਧਦੀ ਜਾਂਦੀ ਹੈ — ਨੇੜੇ ਆਕੇ ਮਾਡਲ ਦੇ ਸੰਦਰਭ ਵਿੰਡੋ ਤੋਂ ਵੱਧ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਨੂੰ **ਸੰਦਰਭ ਸੰਖੇਪ ਪੈਟਰਨ** ਅਤੇ **ਏਜੰਟ ਸਕ੍ਰੈਚਪੈਡ** ਨਾਲ ਸੰਬੋਧਨ ਕਰਦੇ ਹਾਂ ਜੋ ਲਗਾਤਾਰ ਯਾਦ ਰਹਿੰਦਾ ਹੈ।

## ਤੁਸੀਂ ਕੀ ਸਿੱਖੋਗੇ:
1. **ਸੰਦਰਭ ਪ੍ਰਬੰਧਨ ਕਿਉਂ ਮਾਇਨੇ ਰੱਖਦਾ ਹੈ**: ਟੋਕਨ ਸੀਮਾਵਾਂ ਅਤੇ ਸੰਦਰਭ ਵਿੰਡੋਜ਼ ਨੂੰ ਸਮਝਣਾ
2. **ਸੰਦਰਭ-ਸਰਗਰਮ ਏਜੰਟ**: ਉਹ ਏਜੰਟ ਬਣਾਉਣਾ ਜੋ ਆਪਣੀ ਗੱਲਬਾਤ ਦਾ ਸੰਦਰਭ ਖੁਦ ਸੰਭਾਲਦੇ ਹਨ
3. **ਸੰਦਰਭ ਸੰਖੇਪ ਪੈਟਰਨ**: ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਨੂੰ ਸੰਗ੍ਰਹਿਤ ਕਰਨ ਲਈ ਉਪਕਰਨਾਂ ਦੀ ਵਰਤੋਂ
4. **ਏਜੰਟ ਸਕ੍ਰੈਚਪੈਡ**: ਲਗਾਤਾਰ ਯਾਦਾਸ਼ਤ ਜੋ ਸੰਦਰਭ ਘਟਾਉਣ ਤੋਂ ਬਾਅਦ ਵੀ ਬਚੀ ਰਹਿੰਦੀ ਹੈ

## ਲੋੜੀਂਦੇ ਗੁਣ:
- ਵਾਤਾਵਰਣ ਚਲ ਬਦਲੀਆਂ ਨਾਲ Azure OpenAI ਸੈਟਅੱਪ
- ਪਹਿਲਲੇ ਪਾਠਾਂ ਤੋਂ ਬੇਸਿਕ ਏਜੰਟ ਸੰਕਲਪਾਂ ਦੀ ਸਮਝ


## ਸੈੱਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ਸੰਦਰਭ ਪ੍ਰਬੰਧਨ ਕਿਉਂ ਮਹੱਤਵਪੂਰਨ ਹੈ

ਹਰ LLM ਦੀ ਇੱਕ ਸੀਮਿਤ **ਸੰਦਰਭ ਵਿੰਡੋ** ਹੁੰਦੀ ਹੈ — ਇਹ ਇੱਕ درخواست ਵਿੱਚ ਪ੍ਰਕਿਰਿਆ ਕਰ ਸਕਣ ਵਾਲੇ ਟੋਕਨ ਦੀ ਵੱਧ ਤੋਂ ਵੱਧ ਗਿਣਤੀ ਹੈ। ਜਿਵੇਂ ਕਿ ਜਾਣ-ਪਹਚਾਣ ਵਾਲੀ ਗੱਲਬਾਤ ਅੱਗੇ ਵਧਦੀ ਹੈ:

- ਹਰ ਯੂਜ਼ਰ ਸੁਨੇਹੇ ਅਤੇ ਸਹਾਇਕ ਦੇ ਜਵਾਬ ਨਾਲ **ਟੋਕਨ ਗਿਣਤੀ ਰੇਖੀਯਾ ਤੌਰ ਤੇ ਵੱਧਦੀ ਹੈ**।
- **ਪ੍ਰੰਪਟ ਟੋਕਨਾਂ ਦੀ ਲਾਗਤ ਜ਼ਿਆਦਾ ਹੁੰਦੀ ਹੈ** ਕਿਉਂਕਿ ਪੂਰਾ ਇਤਿਹਾਸ ਹਰ ਵਾਰੀ ਦੁਬਾਰਾ ਭੇਜਿਆ ਜਾਂਦਾ ਹੈ।
- ਆਖਿਰਕਾਰ ਗੱਲਬਾਤ **ਸੰਦਰਭ ਵਿੰਡੋ ਤੋਂ ਵੱਧ ਜਾਣਦੀ ਹੈ** ਅਤੇ ਮਾਡਲ ਜਾਂ ਤਾਂ ਕੱਟਦਾ ਹੈ ਜਾਂ ਗਲਤੀ ਕਰਦਾ ਹੈ।

### ਸੰਦਰਭ ਪ੍ਰਬੰਧਨ ਲਈ ਰਣਨੀਤੀਆਂ

| ਰਣਨੀਤੀ | ਇਹ ਕਿਵੇਂ ਕੰਮ ਕਰਦੀ ਹੈ | ਵਪਾਰੀ-ਵਿਰੋਧ |
|---|---|---|
| **ਕੱਟ-ਛਾਂਟ** | ਸਭ ਤੋਂ ਪੁਰਾਣੇ ਸੁਨੇਹੇ ਹਟਾਓ | ਸ਼ੁਰੂਆਤੀ ਸੰਦਰਭ ਖੋ ਜਾਂਦਾ ਹੈ |
| **ਸਾਰ ਸੰਕਲਨ** | ਪੁਰਾਣੇ ਸੁਨੇਹਿਆਂ ਨੂੰ ਇੱਕ ਸਾਰ ਵਿੱਚ ਸੰਕੁੱਚਿਤ ਕਰੋ | ਕੁਝ ਵੇਰਵਾ ਖੋ ਜਾਂਦਾ ਹੈ, ਪਰ ਮੁੱਖ ਬਿੰਦੂ ਬਣੇ ਰਹਿੰਦੇ ਹਨ |
| **ਸਕ੍ਰੈਚਪੈਡ / ਬਾਹਰੀ ਯਾਦاشت** | ਗੱਲਬਾਤ ਤੋਂ ਬਾਹਰ ਮੁੱਖ ਤੱਥ ਸਟੋਰ ਕਰੋ | ਟੂਲ ਕਾਲ ਦੀ ਲੋੜ ਹੈ, ਪਰ ਕਿਸੇ ਵੀ ਕਮੀ ਤੋਂ ਬਾਅਦ ਜੀਉਂਦਾ ਰਹਿੰਦਾ ਹੈ |

ਇਸ ਨੋਟਬੁੱਕ ਵਿੱਚ ਅਸੀਂ **ਸਾਰ ਸੰਕਲਨ** ਨੂੰ **ਸਕ੍ਰੈਚਪੈਡ ਟੂਲ** ਨਾਲ ਜੋੜਦੇ ਹਾਂ ਤਾਂ ਜੋ ਏਜੰਟ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਸੰਕੁੱਚਿਤ ਹੋਣ ਦੇ ਬਾਵਜੂਦ ਲਗਾਤਾਰਤਾ ਬਣਾਈ ਰੱਖ ਸਕੇ।


## ਸੰਦਰਭ-ਸూఝਬੂਝ ਵਾਲਾ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## ਇੱਕ ਲੰਮੇ ਗੱਲਬਾਤ ਦਾ ਅਨੁਕਰਨ ਕਰਨਾ

ਆਓ ਇੱਕ ਕਈ ਵਾਰੀ ਹੋਣ ਵਾਲੀ ਗੱਲਬਾਤ ਦੇ ਜ਼ਰੀਏ ਵੇਖੀਏ ਕਿ ਸੰਦਰਭ ਕਿਵੇਂ ਇਕੱਠਾ ਹੁੰਦਾ ਹੈ। ਏਜੰਟ ਨੂੰ ਮੁੱਖ ਵੇਰਵਿਆਂ (ਪਸੰਦ, ਬਜਟ, ਯਾਤਰਾ ਦੀਆਂ ਤਾਰੀਖਾਂ) ਨੂੰ ਵਾਰੀਦਾਰ ਰੱਖਣਾ ਚਾਹੀਦਾ ਹੈ ਅਤੇ ਲਗਾਤਾਰਤਾ ਨੂੰ ਦਰਸਾਉਣਾ ਚਾਹੀਦਾ ਹੈ।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ਧਿਆਨ ਦਿਓ ਕਿ ਕਿਵੇਂ ਏਜੰਟ ਪਿਛਲੇ ਮੋੜਾਂ ਤੋਂ ਸੰਦਰਭ ਨੂੰ ਬਣਾਈ ਰੱਖਦਾ ਹੈ — ਇਹ ਜਾਪਾਨ, ਸੁਸ਼ੀ, ਮੰਦਰ, ਫੋਟੋਗ੍ਰਾਫੀ, $3000 ਦਾ ਬਜਟ, ਇਕੱਲਾ ਯਾਤਰਾ ਅਤੇ ਅਪ੍ਰੈਲ ਦੇ ਸਮੇਂ ਬਾਰੇ ਜਾਣਦਾ ਹੈ। ਇਕ ਛੋਟੀ ਗੱਲਬਾਤ ਵਿੱਚ ਇਹ ਚੰਗਾ ਕੰਮ ਕਰਦਾ ਹੈ, ਪਰ ਜਿਵੇਂ-ਜਿਵੇਂ ਗੱਲਬਾਤ ਵਧਦੀ ਹੈ ਪੂਰਾ ਇਤਿਹਾਸ ਮੁੜ ਭੇਜਣ ਲਈ ਮਹਿੰਗਾ ਹੁੰਦਾ ਹੈ।

ਆਓ ਗੱਲਬਾਤ ਨੂੰ ਹੋਰ ਮੁਰਾਂ ਨਾਲ ਜਾਰੀ ਰੱਖੀਏ ਤਾਂ ਕਿ ਸੰਦਰਭ ਦੇ ਇੱਕਠੇ ਹੋਣ ਨੂੰ ਵੇਖਿਆ ਜਾ ਸਕੇ:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## ਸੰਦਰਭ ਸਾਰਾਂਸ਼ ਪੈਟਰਨ

ਜਿਵੇਂ ਜਿਵੇਂ ਗੱਲਬਾਤ ਵਧਦੀ ਹੈ, ਅਸੀਂ ਇੱਕ **ਸਾਰਾਂਸ਼ ਸਾਧਨ** ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਇਕੱਠੇ ਹੋਏ ਸੰਦਰਭ ਨੂੰ ਸੰਕੁਚਿਤ ਰੂਪ ਵਿੱਚ ਪ੍ਰਸਤੁਤ ਕਰ ਸਕਦੇ ਹਾਂ। ਏਜੰਟ ਇਸ ਸਾਧਨ ਨੂੰ ਕਾਲ ਕਰਦਾ ਹੈ ਤਾਂ ਜੋ ਮੁੱਖ ਪਸੰਦਾਂ ਨੂੰ ਦਰਜ ਕੀਤਾ ਜਾ ਸਕੇ ਤਾਂ ਜੋ ਜੇ ਪੁਰਾਣੇ ਸੁਨੇਹੇ ਹਟਾ ਦਿੱਤੇ ਜਾਂਦੇ ਹਨ, ਤਾਂ ਵੀ ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਸੁਰੱਖਿਅਤ ਰਹੇ।

ਇਹ ਪੈਟਰਨ ਜ਼ਿਆਦਾ ਬਹੁਤਰੀਤ ਇਤਿਹਾਸ ਘਟਾਉਣ ਲਈ ਬਿਲਡਿੰਗ ਬਲਾਕ ਹੈ:
1. ਏਜੰਟ ਗੱਲਬਾਤ ਤੋਂ ਮੁੱਖ ਤੱਥਾਂ ਦੀ ਪਹਿਚਾਣ ਕਰਦਾ ਹੈ
2. ਇਹ ਸਾਰਾਂਸ਼ ਸਾਧਨ ਨੂੰ ਕਾਲ ਕਰਦਾ ਹੈ ਤਾਂ ਜੋ ਉਹਨਾਂ ਨੂੰ ਸਥਿਰ ਕੀਤਾ ਜਾ ਸਕੇ
3. ਪੁਰਾਣੇ ਸੁਨੇਹੇ ਸੁਰੱਖਿਅਤ ਤੌਰ 'ਤੇ ਹਟਾਏ ਜਾ ਸਕਦੇ ਹਨ ਕਿਉਂਕਿ ਸਾਰਾਂਸ਼ਜ਼ ਜੋ ਮਹੱਤਵਪੂਰਨ ਹੈ ਉਸ ਨੂੰ ਕੈਦ ਕਰਦਾ ਹੈ

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ `summarize_preferences` ਸਾਧਨ ਦੀ ਪਰਿਭਾਸ਼ਾ ਕਰਦੇ ਹਾਂ ਜਿਸਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ ਤਾਂ ਜੋ ਉਹ ਜੋ ਕੁਝ ਸਿੱਖਿਆ ਹੈ ਉਸ ਦਾ ਸੰਕੁਚਿਤ ਸਾਰਾਂਸ਼ ਦਰਜ ਕੀਤਾ ਜਾ ਸਕੇ।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ ਲੰਬੇ ਸਮੇਂ ਚੱਲ ਰਹੀਆਂ ਏਜੰਟ ਗੱਲਬਾਤਾਂ ਵਿੱਚ ਸੰਦਰਭ ਕਿਵੇਂ ਪ੍ਰਬੰਧਿਤ ਕਰਨਾ ਸਿੱਖਿਆ:

### ਮੁੱਖ ਧਾਰਣਾਵਾਂ
- **ਸੰਦਰਭ ਵਿੰਡੋਆਂ ਸੀਮਿਤ ਹੁੰਦੀਆਂ ਹਨ** — ਗੱਲਬਾਤ ਦੇ ਇਤਿਹਾਸ ਵਿੱਚ ਹਰ ਟੋਕਨ ਦੀ ਲਾਗਤ ਪੈਂਦੀ ਹੈ ਅਤੇ ਇਹ ਸੀਮਾ ਵੱਲ ਗਿਣਤੀ ਵਿੱਚ ਆਉਂਦਾ ਹੈ।
- **ਸੰਖੇਪ ਕਰਨ ਵਾਲੇ ਸੰਦ** ਏਜੰਟ ਨੂੰ ਇਕੱਠੇ ਕੀਤੇ ਸੰਦਰਭ ਨੂੰ ਸੰਖੇਪ ਰੂਪਾਂਤਰਾਂ ਵਿੱਚ ਕੱਦ ਕੇ ਟੋਕਨ ਦੀ ਵਰਤੋਂ ਘਟਾਉਂਦੇ ਹਨ ਅਤੇ ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਨੂੰ ਬਚਾਉਂਦੇ ਹਨ।
- **ਏਜੰਟ ਸਕ੍ਰੈਚਪੈਡਸ** ਇਕ ਸਥਾਈ ਬਾਹਰੀ ਯਾਦਦਾਸ਼ਤ ਪ੍ਰਦਾਨ ਕਰਦੇ ਹਨ ਜੋ ਕਿਸੇ ਵੀ ਗੱਲਬਾਤ ਸਹਿਜੋੜ ਨੂੰ ਬਰਕਰਾਰ ਰੱਖਦਾ ਹੈ।

### ਤੁਸੀਂ ਜੋ ਬਣਾਇਆ
- ਇੱਕ **ਸੰਦਰਭ-ਸੂਚਿਤ ਏਜੰਟ** ਜੋ ਬਹੁ-ਚਰਣ ਗੱਲਬਾਤਾਂ ਵਿੱਚ ਲਗਾਤਾਰਤਾ ਬਣਾਈ ਰੱਖਦਾ ਹੈ
- ਇੱਕ **ਸੰਖੇਪ ਕਰਨ ਵਾਲਾ ਸੰਦ** (`summarize_preferences`) ਜੋ ਮੁੱਖ ਉਪਭੋਗਤਾ ਵਿਸਥਾਰਾਂ ਨੂੰ ਸੰਖੇਪ ਰੂਪ ਵਿੱਚ ਦਰਜ ਕਰਦਾ ਹੈ
- ਇੱਕ **ਬਹੁ-ਚਰਣ ਗੱਲਬਾਤ** ਜੋ ਸੰਦਰਭ ਸਨਭਾਲ ਅਤੇ ਬਦਲਾਅ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ

### ਅਸਲੀ ਦੁਨੀਆ ਦੇ ਐਪਲੀਕੇਸ਼ਨ
- **ਕਸਟਮਰ ਸਰਵਿਸ ਬੋਟਸ**: ਲੰਬੀਆਂ ਸਹਾਇਤਾ ਸੈਸ਼ਨਾਂ ਦੌਰਾਨ ਪਸੰਦਾਂ ਨੂੰ ਯਾਦ ਰੱਖਣਾ
- **ਵੈੱਅਕਤੀਕ ਸਹਾਇਕ**: ਸੰਦਰਭ ਨੂੰ ਫਿਰ ਤੋਂ ਸਮਝਾਏ ਬਿਨਾਂ ਚੱਲ ਰਹੀਆਂ ਪ੍ਰੋਜੈਕਟਾਂ ਦਾ ਪਿਛਾ ਕਰਨਾ
- **ਸਿੱਖਿਆ ਅਧਿਆਪਕ**: ਕਈ ਮੁਲਾਕਾਤਾਂ ਦੌਰਾਨ ਵਿਦਿਆਰਥੀ ਦੀ ਪ੍ਰਗਟੀ ਨੂੰ ਬਣਾਈ ਰੱਖਣਾ

### ਅਗਲੇ ਕਦਮ
- ਫਾਈਲ-ਆਧਾਰਿਤ ਸਥਿਰਤਾ ਨਾਲ ਇੱਕ ਪੂਰਾ ਸਕ੍ਰੈਚਪੈਡ ਔਜ਼ਾਰ ਲਾਗੂ ਕਰੋ
- ਸੰਖੇਪ ਕਰਨ ਤੋਂ ਬਾਅਦ ਸਵੈਚਾਲਿਤ ਇਤਿਹਾਸ ਛਾਂਟਨਾ ਸ਼ਾਮਿਲ ਕਰੋ
- ਸੈਮਾਂਟਿਕ ਯਾਦਦਾਸ਼ਤ ਖੋਜ ਲਈ ਵੈਕਟਰ ਡੇਟਾਬੇਸਾਂ ਨਾਲ ਜੋੜੋ
- ਐਜੰਟ ਬਣਾਓ ਜੋ ਪੂਰੇ ਸੰਦਰਭ ਨਾਲ ਕੁਝ ਦਿਨ ਬਾਅਦ ਗੱਲਬਾਤ ਮੁੜ ਸ਼ੁਰੂ ਕਰ ਸਕਦੇ ਹਨ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
